In [1]:
"""
================================================================================
Tesla Direction Classification - EXTENDED MODEL ZOO + RICH COMPARISONS
================================================================================
Models (toggle each via RUN_* flags):

  Baselines:        Always-Up, Majority-Class, Persistence, Logistic Regression
  Tree-based:       Random Forest, XGBoost, LightGBM
  Sequence (small): 1D CNN, GRU, LSTM
  Sequence (heavy): CNN-LSTM, Transformer encoder, TCN, TFT, N-BEATS

Features used: close return + 3 precomputed sentiment scores (+ lags 1,2,3)
No-leakage design: features at time t use only info through t-1.

Auto-generated outputs:
  - results_by_fold.csv         (one row per model x fold)
  - results_summary.csv         (mean +/- std per model)
  - 01_metric_bars.png          (all 6 metrics, bar charts with random baseline)
  - 02_radar_top_models.png     (multi-metric radar of top 5)
  - 03_per_fold_lines.png       (BalAcc and MCC across folds)
  - 04_boxplots.png             (distribution of each metric across folds)
  - 05_ranking_heatmap.png      (rank of each model per metric)
  - 06_confusion_matrices.png   (best fold per top model)
  - 07_roc_curves.png           (ROC for top 5 on last fold)
  - 08_calibration.png          (calibration curves)
  - 09_feature_importance.png   (LogReg + RF/XGB/LGBM)
================================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Dense, LSTM, GRU, Dropout, LayerNormalization, BatchNormalization,
    MultiHeadAttention, GlobalAveragePooling1D, Add, Subtract, Flatten,
    Conv1D, MaxPooling1D, SpatialDropout1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef,
    roc_auc_score, brier_score_loss,
    confusion_matrix, roc_curve, calibration_curve
)
import keras_tuner as kt

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.makedirs("plots", exist_ok=True)

# ============================================================================
# 1. CONFIG
# ============================================================================
CSV_PATH    = "brian_news_stocks_tesla_2023_processses_with_agg.csv"
TARGET_COL  = "close"
TIME_COL    = "timestamp"
LOOKBACK    = 20
N_SPLITS    = 5
MAX_TRIALS  = 10
TUNE_EPOCHS = 25
FIT_EPOCHS  = 100
BATCH_SIZE  = 32

RUN_NAIVE       = True
RUN_LOGREG      = True
RUN_RF          = True
RUN_XGB         = True
RUN_LGBM        = True
RUN_CNN         = True
RUN_GRU         = True
RUN_LSTM        = True
RUN_CNN_LSTM    = True
RUN_TRANSFORMER = True
RUN_TCN         = True
RUN_TFT         = True
RUN_NBEATS      = True


# ============================================================================
# 2. FEATURE ENGINEERING (no leakage)
# ============================================================================
def build_features(df, lags=(1, 2, 3)):
    df = df.sort_values(TIME_COL).reset_index(drop=True).copy()
    df["return"] = df[TARGET_COL].pct_change().fillna(0)

    score_features = [
        "avg_score_of_normalized_min",
        "avg_score_of_normalized_max",
        "avg_score_of_normalized_avg",
    ]
    for c in score_features:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    for lag in lags:
        for c in score_features:
            df[f"{c}_lag{lag}"] = df[c].shift(lag).fillna(0)

    feature_cols = ["return"] + score_features + [
        f"{c}_lag{lag}" for lag in lags for c in score_features
    ]

    # Shift features by 1: at time t we only know features through t-1
    for c in feature_cols:
        df[c] = df[c].shift(1)
    df["target"] = (df[TARGET_COL] > df[TARGET_COL].shift(1)).astype(int)
    df = df.dropna(subset=feature_cols + ["target"]).reset_index(drop=True)
    return df[feature_cols], df["target"].values.astype(int), feature_cols


def create_windows(features_df, target, look_back=LOOKBACK):
    arr = features_df.values.astype("float32")
    X, y = [], []
    for i in range(len(arr) - look_back):
        X.append(arr[i : i + look_back])
        y.append(target[i + look_back])
    return np.array(X), np.array(y, dtype="int32")


# ============================================================================
# 3. METRICS + UTILITIES
# ============================================================================
METRIC_KEYS = ["Accuracy", "BalancedAcc", "F1", "MCC", "ROC-AUC", "Brier"]
HIGHER_IS_BETTER = {"Accuracy": True, "BalancedAcc": True, "F1": True,
                    "MCC": True, "ROC-AUC": True, "Brier": False}


def classification_metrics(y_true, y_pred_proba, threshold=0.5):
    y_true = np.asarray(y_true).flatten()
    y_proba = np.asarray(y_pred_proba).flatten()
    y_pred = (y_proba >= threshold).astype(int)
    metrics = {
        "Accuracy":    accuracy_score(y_true, y_pred),
        "BalancedAcc": balanced_accuracy_score(y_true, y_pred),
        "F1":          f1_score(y_true, y_pred, zero_division=0),
        "MCC":         matthews_corrcoef(y_true, y_pred),
        "Brier":       brier_score_loss(y_true, y_proba),
    }
    metrics["ROC-AUC"] = (roc_auc_score(y_true, y_proba)
                          if len(np.unique(y_true)) > 1 else np.nan)
    return metrics


def find_best_threshold(y_true, y_proba):
    best_t, best_score = 0.5, -1
    for t in np.linspace(0.3, 0.7, 41):
        score = balanced_accuracy_score(y_true, (y_proba >= t).astype(int))
        if score > best_score:
            best_score, best_t = score, t
    return best_t


def summarize(metrics_list, name=""):
    print(f"\n=== {name} (mean +/- std across {len(metrics_list)} folds) ===")
    for k in metrics_list[0].keys():
        vals = [m[k] for m in metrics_list if not np.isnan(m[k])]
        if vals:
            print(f"  {k:13s}: {np.mean(vals):7.4f} +/- {np.std(vals):.4f}")


def scale_windows(X_train, X_test):
    n, lb, f = X_train.shape
    sc = StandardScaler().fit(X_train.reshape(-1, f))
    X_tr = sc.transform(X_train.reshape(-1, f)).reshape(n, lb, f)
    X_te = sc.transform(X_test.reshape(-1, f)).reshape(X_test.shape)
    return X_tr, X_te, sc


# ============================================================================
# 4. LOAD DATA
# ============================================================================
df_raw = pd.read_csv(CSV_PATH)
print(f"Raw data: {len(df_raw)} rows, {df_raw.shape[1]} cols")
feats_df, y_all, feature_cols = build_features(df_raw)
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Labels: up={y_all.sum()} ({y_all.mean()*100:.1f}%) "
      f"down={(1-y_all).sum()} ({(1-y_all.mean())*100:.1f}%)")
X_all, y_seq = create_windows(feats_df, y_all, LOOKBACK)
N_FEATURES = X_all.shape[-1]
print(f"Windows: {X_all.shape}, labels: {y_seq.shape}")


# ============================================================================
# 5. STORAGE FOR DETAILED RESULTS (used by plotting)
# ============================================================================
all_fold_metrics = {}        # name -> list of dicts (per fold)
all_fold_predictions = {}    # name -> list of (y_true, y_proba) per fold
feature_importances = {}     # name -> dict {feature: importance}


def store_predictions(name, y_true, y_proba):
    all_fold_predictions.setdefault(name, []).append(
        (np.asarray(y_true).flatten(), np.asarray(y_proba).flatten())
    )


# ============================================================================
# 6. NAIVE BASELINES
# ============================================================================
def evaluate_naive_baselines(X, y):
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    results = {"Always-Up": [], "Majority": [], "Persistence": []}
    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X), 1):
        y_tr, y_te = y[tr_idx], y[te_idx]
        X_te = X[te_idx]
        # Always-Up
        proba = np.ones(len(y_te))
        results["Always-Up"].append(classification_metrics(y_te, proba))
        store_predictions("Always-Up", y_te, proba)
        # Majority
        maj = int(y_tr.mean() >= 0.5)
        proba = np.full(len(y_te), maj, dtype=float)
        results["Majority"].append(classification_metrics(y_te, proba))
        store_predictions("Majority", y_te, proba)
        # Persistence: same direction as last bar's return
        proba = (X_te[:, -1, 0] > 0).astype(float)
        results["Persistence"].append(classification_metrics(y_te, proba))
        store_predictions("Persistence", y_te, proba)
    return results


# ============================================================================
# 7. SHARED CV LOOP FOR SKLEARN-STYLE MODELS (flatten window)
# ============================================================================
def run_sklearn_model(make_model, name, X, y, store_importance_attr=None):
    print(f"\n>>> {name}")
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    fold_metrics = []
    importances = []
    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X), 1):
        X_tr_w, X_te_w = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]
        X_tr = X_tr_w.reshape(len(X_tr_w), -1)
        X_te = X_te_w.reshape(len(X_te_w), -1)
        sc = StandardScaler().fit(X_tr)
        X_tr, X_te = sc.transform(X_tr), sc.transform(X_te)

        clf = make_model()
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_te)[:, 1]
        m = classification_metrics(y_te, proba)
        fold_metrics.append(m)
        store_predictions(name, y_te, proba)

        if store_importance_attr and hasattr(clf, store_importance_attr):
            imp = getattr(clf, store_importance_attr)
            if imp.ndim > 1:
                imp = np.abs(imp).mean(axis=0)
            importances.append(imp)

        print(f"  {name} fold {fold}: BalAcc={m['BalancedAcc']:.4f} "
              f"F1={m['F1']:.4f} MCC={m['MCC']:.4f}")

    if importances:
        avg_imp = np.mean(importances, axis=0).reshape(LOOKBACK, N_FEATURES)
        feature_importances[name] = dict(zip(feature_cols,
                                              np.abs(avg_imp).mean(axis=0)))
    return fold_metrics


# ============================================================================
# 8. KERAS HYPERMODELS
# ============================================================================
def build_lstm(hp):
    m = Sequential()
    n = hp.Int("num_layers", 1, 2)
    for i in range(n):
        u = hp.Int(f"units_{i}", 16, 64, step=16)
        d = hp.Float(f"dropout_{i}", 0.1, 0.5, step=0.1)
        rs = (i < n - 1)
        if i == 0:
            m.add(LSTM(u, return_sequences=rs, input_shape=(LOOKBACK, N_FEATURES)))
        else:
            m.add(LSTM(u, return_sequences=rs))
        m.add(Dropout(d))
    m.add(Dense(1, activation="sigmoid"))
    lr = hp.Choice("lr", [1e-2, 1e-3, 1e-4])
    m.compile(tf.keras.optimizers.Adam(lr), "binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return m


def build_gru(hp):
    m = Sequential()
    n = hp.Int("num_layers", 1, 2)
    for i in range(n):
        u = hp.Int(f"units_{i}", 16, 64, step=16)
        d = hp.Float(f"dropout_{i}", 0.1, 0.5, step=0.1)
        rs = (i < n - 1)
        if i == 0:
            m.add(GRU(u, return_sequences=rs, input_shape=(LOOKBACK, N_FEATURES)))
        else:
            m.add(GRU(u, return_sequences=rs))
        m.add(Dropout(d))
    m.add(Dense(1, activation="sigmoid"))
    lr = hp.Choice("lr", [1e-2, 1e-3, 1e-4])
    m.compile(tf.keras.optimizers.Adam(lr), "binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return m


def build_cnn(hp):
    """1D CNN: fast and competitive with small data."""
    m = Sequential()
    n = hp.Int("num_conv_layers", 1, 2)
    for i in range(n):
        f = hp.Int(f"filters_{i}", 16, 64, step=16)
        k = hp.Choice(f"kernel_{i}", [3, 5])
        if i == 0:
            m.add(Conv1D(f, k, activation="relu", padding="same",
                         input_shape=(LOOKBACK, N_FEATURES)))
        else:
            m.add(Conv1D(f, k, activation="relu", padding="same"))
        m.add(SpatialDropout1D(hp.Float(f"sdrop_{i}", 0.0, 0.3, step=0.1)))
    m.add(GlobalAveragePooling1D())
    m.add(Dense(hp.Int("dense_units", 16, 64, step=16), activation="relu"))
    m.add(Dropout(hp.Float("dropout", 0.1, 0.5, step=0.1)))
    m.add(Dense(1, activation="sigmoid"))
    lr = hp.Choice("lr", [1e-2, 1e-3, 1e-4])
    m.compile(tf.keras.optimizers.Adam(lr), "binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return m


def build_cnn_lstm(hp):
    """Conv1D feature extractor -> LSTM."""
    m = Sequential()
    f = hp.Int("filters", 16, 64, step=16)
    k = hp.Choice("kernel", [3, 5])
    m.add(Conv1D(f, k, activation="relu", padding="same",
                 input_shape=(LOOKBACK, N_FEATURES)))
    m.add(MaxPooling1D(2))
    m.add(SpatialDropout1D(hp.Float("sdrop", 0.0, 0.3, step=0.1)))
    m.add(LSTM(hp.Int("lstm_units", 16, 64, step=16)))
    m.add(Dropout(hp.Float("dropout", 0.1, 0.5, step=0.1)))
    m.add(Dense(1, activation="sigmoid"))
    lr = hp.Choice("lr", [1e-2, 1e-3, 1e-4])
    m.compile(tf.keras.optimizers.Adam(lr), "binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return m


def build_transformer(hp):
    """Pure-attention encoder."""
    units   = hp.Int("units", 16, 64, step=16)
    heads   = hp.Choice("heads", [2, 4])
    dropout = hp.Float("dropout", 0.1, 0.4, step=0.1)
    lr      = hp.Choice("lr", [1e-3, 5e-4, 1e-4])

    inp = Input(shape=(LOOKBACK, N_FEATURES))
    x = Dense(units)(inp)  # project to model dim
    attn = MultiHeadAttention(num_heads=heads, key_dim=units)(x, x)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization()(Add()([x, attn]))
    ff = Dense(units * 2, activation="relu")(x)
    ff = Dense(units)(ff)
    ff = Dropout(dropout)(ff)
    x = LayerNormalization()(Add()([x, ff]))
    x = GlobalAveragePooling1D()(x)
    out = Dense(1, activation="sigmoid")(x)
    model = Model(inp, out)
    model.compile(tf.keras.optimizers.Adam(lr), "binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model


def build_tcn(hp):
    """Temporal Convolutional Network: dilated causal convolutions."""
    filters  = hp.Int("filters", 16, 64, step=16)
    n_blocks = hp.Int("n_blocks", 2, 4)
    dropout  = hp.Float("dropout", 0.1, 0.4, step=0.1)
    lr       = hp.Choice("lr", [1e-3, 5e-4, 1e-4])

    inp = Input(shape=(LOOKBACK, N_FEATURES))
    x = inp
    for i in range(n_blocks):
        dilation = 2 ** i
        residual = x
        x = Conv1D(filters, 3, padding="causal", dilation_rate=dilation,
                   activation="relu")(x)
        x = SpatialDropout1D(dropout)(x)
        x = Conv1D(filters, 3, padding="causal", dilation_rate=dilation,
                   activation="relu")(x)
        x = SpatialDropout1D(dropout)(x)
        if residual.shape[-1] != filters:
            residual = Conv1D(filters, 1, padding="same")(residual)
        x = Add()([residual, x])
    x = GlobalAveragePooling1D()(x)
    out = Dense(1, activation="sigmoid")(x)
    model = Model(inp, out)
    model.compile(tf.keras.optimizers.Adam(lr), "binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model


def gated_residual_network(x, units, dropout=0.1):
    h = Dense(units, activation="elu")(x)
    h = Dense(units)(h)
    h = Dropout(dropout)(h)
    gate = Dense(units, activation="sigmoid")(x)
    if x.shape[-1] != units:
        x = Dense(units)(x)
    return LayerNormalization()(Add()([x, h * gate]))


def build_tft(hp):
    units   = hp.Int("units", 16, 64, step=16)
    heads   = hp.Choice("heads", [2, 4])
    dropout = hp.Float("dropout", 0.1, 0.4, step=0.1)
    lr      = hp.Choice("lr", [1e-3, 5e-4, 1e-4])

    inp = Input(shape=(LOOKBACK, N_FEATURES))
    enc = LSTM(units, return_sequences=True)(inp)
    enc = gated_residual_network(enc, units, dropout)
    attn = MultiHeadAttention(num_heads=heads, key_dim=units)(enc, enc)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization()(Add()([enc, attn]))
    x = gated_residual_network(x, units, dropout)
    x = GlobalAveragePooling1D()(x)
    out = Dense(1, activation="sigmoid")(x)
    model = Model(inp, out)
    model.compile(tf.keras.optimizers.Adam(lr), "binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model


def nbeats_block(x, units, flat_size):
    h = x
    for _ in range(4):
        h = Dense(units, activation="relu")(h)
    backcast = Dense(flat_size, activation="linear")(h)
    forecast = Dense(units, activation="linear")(h)
    return backcast, forecast


def build_nbeats(hp):
    units    = hp.Int("units", 32, 128, step=32)
    n_blocks = hp.Int("n_blocks", 2, 4)
    dropout  = hp.Float("dropout", 0.1, 0.4, step=0.1)
    lr       = hp.Choice("lr", [1e-3, 5e-4, 1e-4])
    flat_size = LOOKBACK * N_FEATURES

    inp = Input(shape=(LOOKBACK, N_FEATURES))
    flat = Flatten()(inp)
    residual = flat
    forecast_sum = None
    for _ in range(n_blocks):
        b, fc = nbeats_block(residual, units, flat_size)
        residual = Subtract()([residual, b])
        forecast_sum = fc if forecast_sum is None else Add()([forecast_sum, fc])
    x = Dropout(dropout)(forecast_sum)
    out = Dense(1, activation="sigmoid")(x)
    model = Model(inp, out)
    model.compile(tf.keras.optimizers.Adam(lr), "binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model


# ============================================================================
# 9. SHARED KERAS WALK-FORWARD RUNNER
# ============================================================================
def run_keras_model(builder, name, X, y):
    print(f"\n>>> {name}")
    split = int(len(X) * 0.7)
    cw_arr = compute_class_weight("balanced",
                                  classes=np.unique(y[:split]), y=y[:split])
    cw = {i: w for i, w in enumerate(cw_arr)}

    X_tune, _, _ = scale_windows(X[:split], X[:split])
    val_split = int(len(X_tune) * 0.8)
    tuner = kt.BayesianOptimization(
        builder, objective=kt.Objective("val_auc", direction="max"),
        max_trials=MAX_TRIALS, directory="kt_dir_zoo",
        project_name=f"{name}_zoo", overwrite=True, seed=SEED,
    )
    es = EarlyStopping(monitor="val_auc", mode="max", patience=8,
                       min_delta=1e-4, restore_best_weights=True)
    tuner.search(X_tune[:val_split], y[:split][:val_split],
                 validation_data=(X_tune[val_split:], y[:split][val_split:]),
                 epochs=TUNE_EPOCHS, class_weight=cw,
                 callbacks=[es], verbose=0)
    best_hp = tuner.get_best_hyperparameters(1)[0]
    print(f"  Best HPs: {best_hp.values}")

    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    fold_metrics = []
    for fold, (tr_idx, te_idx) in enumerate(tscv.split(X), 1):
        X_tr_raw, X_te_raw = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]
        X_tr, X_te, _ = scale_windows(X_tr_raw, X_te_raw)

        val_split = int(len(X_tr) * 0.85)
        X_t, X_v = X_tr[:val_split], X_tr[val_split:]
        y_t, y_v = y_tr[:val_split], y_tr[val_split:]
        cw_arr = compute_class_weight("balanced",
                                      classes=np.unique(y_t), y=y_t)
        cw = {i: w for i, w in enumerate(cw_arr)}

        model = builder(best_hp)
        callbacks = [
            EarlyStopping(monitor="val_auc", mode="max", patience=15,
                          min_delta=1e-4, restore_best_weights=True),
            ReduceLROnPlateau(monitor="val_loss", patience=6, factor=0.5),
        ]
        model.fit(X_t, y_t, epochs=FIT_EPOCHS, batch_size=BATCH_SIZE,
                  validation_data=(X_v, y_v),
                  class_weight=cw, callbacks=callbacks, verbose=0)

        val_proba = model.predict(X_v, verbose=0).flatten()
        best_t = find_best_threshold(y_v, val_proba)
        test_proba = model.predict(X_te, verbose=0).flatten()
        m = classification_metrics(y_te, test_proba, threshold=best_t)
        m["Threshold"] = best_t
        fold_metrics.append(m)
        store_predictions(name, y_te, test_proba)
        print(f"  {name} fold {fold}: BalAcc={m['BalancedAcc']:.4f} "
              f"F1={m['F1']:.4f} MCC={m['MCC']:.4f} thr={best_t:.2f}")
    return fold_metrics


# ============================================================================
# 10. PLOTTING SUITE
# ============================================================================
def plot_metric_bars(df_summary):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for ax, metric in zip(axes, METRIC_KEYS):
        means = df_summary[f"{metric}_mean"].values
        stds  = df_summary[f"{metric}_std"].values
        order = np.argsort(means if HIGHER_IS_BETTER[metric] else -means)[::-1]
        names = df_summary["Model"].values[order]
        ax.barh(range(len(names)), means[order], xerr=stds[order],
                color="steelblue", alpha=0.85, capsize=3)
        ax.set_yticks(range(len(names)))
        ax.set_yticklabels(names, fontsize=9)
        ax.invert_yaxis()
        ax.set_title(f"{metric} ({'higher better' if HIGHER_IS_BETTER[metric] else 'lower better'})")
        # Random-baseline reference lines
        if metric == "BalancedAcc":
            ax.axvline(0.5, color="red", ls="--", lw=1, label="Random")
            ax.legend()
        elif metric == "MCC":
            ax.axvline(0, color="red", ls="--", lw=1, label="Random")
            ax.legend()
        elif metric == "ROC-AUC":
            ax.axvline(0.5, color="red", ls="--", lw=1, label="Random")
            ax.legend()
    plt.suptitle("Model Comparison - All Metrics (mean +/- std across folds)",
                 fontsize=14, y=1.00)
    plt.tight_layout()
    plt.savefig("plots/01_metric_bars.png", dpi=120, bbox_inches="tight")
    plt.close()


def plot_radar_top_models(df_summary, top_n=5):
    # Pick by MCC (most informative single metric for imbalanced binary)
    top = df_summary.nlargest(top_n, "MCC_mean")
    metrics_for_radar = ["Accuracy", "BalancedAcc", "F1", "MCC", "ROC-AUC"]
    # Normalize each metric to [0,1] across the top_n for visualization
    norm_data = {}
    for m in metrics_for_radar:
        vals = top[f"{m}_mean"].values
        rng = vals.max() - vals.min() if vals.max() != vals.min() else 1
        norm_data[m] = (vals - vals.min()) / rng
    angles = np.linspace(0, 2 * np.pi, len(metrics_for_radar), endpoint=False).tolist()
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    for i, model in enumerate(top["Model"].values):
        vals = [norm_data[m][i] for m in metrics_for_radar]
        vals += vals[:1]
        ax.plot(angles, vals, label=model, lw=2)
        ax.fill(angles, vals, alpha=0.1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics_for_radar)
    ax.set_title(f"Top {top_n} Models - Normalized Metrics", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig("plots/02_radar_top_models.png", dpi=120, bbox_inches="tight")
    plt.close()


def plot_per_fold_lines(by_fold_df):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, metric, title in zip(axes, ["BalancedAcc", "MCC"],
                                  ["Balanced Accuracy", "MCC"]):
        for model in by_fold_df["Model"].unique():
            sub = by_fold_df[by_fold_df["Model"] == model]
            ax.plot(sub["Fold"], sub[metric], marker="o", label=model, alpha=0.7)
        ax.set_title(f"{title} per Fold")
        ax.set_xlabel("Fold")
        ax.set_ylabel(metric)
        if metric == "BalancedAcc":
            ax.axhline(0.5, color="red", ls="--", alpha=0.5)
        else:
            ax.axhline(0, color="red", ls="--", alpha=0.5)
        ax.legend(fontsize=7, loc="best")
        ax.grid(alpha=0.3)
    plt.suptitle("Stability Across Folds (a model can win on average but lose on stability)",
                 fontsize=12)
    plt.tight_layout()
    plt.savefig("plots/03_per_fold_lines.png", dpi=120, bbox_inches="tight")
    plt.close()


def plot_boxplots(by_fold_df):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for ax, metric in zip(axes, METRIC_KEYS):
        order = (by_fold_df.groupby("Model")[metric]
                 .median()
                 .sort_values(ascending=not HIGHER_IS_BETTER[metric])
                 .index.tolist())
        data = [by_fold_df[by_fold_df["Model"] == m][metric].values for m in order]
        bp = ax.boxplot(data, labels=order, patch_artist=True, showmeans=True)
        for p in bp["boxes"]:
            p.set_facecolor("lightsteelblue")
        ax.set_title(metric)
        ax.tick_params(axis="x", rotation=45)
        if metric in ("BalancedAcc", "ROC-AUC"):
            ax.axhline(0.5, color="red", ls="--", alpha=0.5)
        elif metric == "MCC":
            ax.axhline(0, color="red", ls="--", alpha=0.5)
    plt.suptitle("Metric Distribution Across Folds", fontsize=14)
    plt.tight_layout()
    plt.savefig("plots/04_boxplots.png", dpi=120, bbox_inches="tight")
    plt.close()


def plot_ranking_heatmap(df_summary):
    rank_df = pd.DataFrame({"Model": df_summary["Model"]})
    for m in METRIC_KEYS:
        ascending = not HIGHER_IS_BETTER[m]
        rank_df[m] = df_summary[f"{m}_mean"].rank(ascending=ascending).astype(int)
    rank_df = rank_df.set_index("Model")
    rank_df["AvgRank"] = rank_df.mean(axis=1)
    rank_df = rank_df.sort_values("AvgRank")
    fig, ax = plt.subplots(figsize=(10, max(6, 0.4 * len(rank_df))))
    im = ax.imshow(rank_df.values, cmap="RdYlGn_r", aspect="auto")
    ax.set_xticks(range(len(rank_df.columns)))
    ax.set_xticklabels(rank_df.columns, rotation=30)
    ax.set_yticks(range(len(rank_df)))
    ax.set_yticklabels(rank_df.index)
    for i in range(len(rank_df)):
        for j in range(len(rank_df.columns)):
            ax.text(j, i, f"{rank_df.values[i,j]:.1f}",
                    ha="center", va="center", color="black", fontsize=9)
    ax.set_title("Model Ranking by Metric (1 = best, lower is better)")
    plt.colorbar(im, ax=ax, label="Rank")
    plt.tight_layout()
    plt.savefig("plots/05_ranking_heatmap.png", dpi=120, bbox_inches="tight")
    plt.close()


def plot_confusion_matrices(df_summary, top_n=6):
    top = df_summary.nlargest(top_n, "MCC_mean")["Model"].values
    n_cols = 3
    n_rows = int(np.ceil(top_n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = axes.flatten()
    for ax, model in zip(axes, top):
        if model not in all_fold_predictions:
            ax.axis("off"); continue
        # Aggregate predictions across all folds
        y_true_all = np.concatenate([yt for yt, _ in all_fold_predictions[model]])
        y_proba_all = np.concatenate([yp for _, yp in all_fold_predictions[model]])
        y_pred = (y_proba_all >= 0.5).astype(int)
        cm = confusion_matrix(y_true_all, y_pred)
        im = ax.imshow(cm, cmap="Blues")
        ax.set_title(f"{model}\nTotal samples: {len(y_true_all)}")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["Down", "Up"])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["Down", "Up"])
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, cm[i, j], ha="center", va="center",
                        color="white" if cm[i, j] > cm.max()/2 else "black",
                        fontsize=12)
    for ax in axes[len(top):]:
        ax.axis("off")
    plt.suptitle(f"Confusion Matrices - Top {top_n} Models (all folds aggregated)", fontsize=13)
    plt.tight_layout()
    plt.savefig("plots/06_confusion_matrices.png", dpi=120, bbox_inches="tight")
    plt.close()


def plot_roc_curves(df_summary, top_n=5):
    top = df_summary.nlargest(top_n, "MCC_mean")["Model"].values
    fig, ax = plt.subplots(figsize=(8, 8))
    for model in top:
        if model not in all_fold_predictions:
            continue
        y_true_all = np.concatenate([yt for yt, _ in all_fold_predictions[model]])
        y_proba_all = np.concatenate([yp for _, yp in all_fold_predictions[model]])
        if len(np.unique(y_true_all)) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_true_all, y_proba_all)
        auc = roc_auc_score(y_true_all, y_proba_all)
        ax.plot(fpr, tpr, lw=2, label=f"{model} (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curves - Top {top_n} Models (folds aggregated)")
    ax.legend(loc="lower right"); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("plots/07_roc_curves.png", dpi=120, bbox_inches="tight")
    plt.close()


def plot_calibration(df_summary, top_n=5):
    top = df_summary.nlargest(top_n, "MCC_mean")["Model"].values
    fig, ax = plt.subplots(figsize=(8, 8))
    for model in top:
        if model not in all_fold_predictions:
            continue
        y_true_all = np.concatenate([yt for yt, _ in all_fold_predictions[model]])
        y_proba_all = np.concatenate([yp for _, yp in all_fold_predictions[model]])
        if len(np.unique(y_proba_all)) < 5:
            continue  # skip naive baselines that always predict 0/1
        try:
            prob_true, prob_pred = calibration_curve(y_true_all, y_proba_all,
                                                     n_bins=8, strategy="quantile")
            ax.plot(prob_pred, prob_true, marker="o", lw=2, label=model)
        except Exception:
            pass
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect calibration")
    ax.set_xlabel("Predicted probability"); ax.set_ylabel("Observed frequency")
    ax.set_title("Calibration Curves (probabilities vs reality)")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("plots/08_calibration.png", dpi=120, bbox_inches="tight")
    plt.close()


def plot_feature_importance():
    if not feature_importances:
        return
    fig, axes = plt.subplots(1, len(feature_importances),
                              figsize=(6 * len(feature_importances), 6),
                              squeeze=False)
    axes = axes[0]
    for ax, (name, imp_dict) in zip(axes, feature_importances.items()):
        sorted_items = sorted(imp_dict.items(), key=lambda x: x[1], reverse=True)
        feats, vals = zip(*sorted_items)
        ax.barh(range(len(feats)), vals, color="seagreen", alpha=0.85)
        ax.set_yticks(range(len(feats)))
        ax.set_yticklabels(feats, fontsize=8)
        ax.invert_yaxis()
        ax.set_title(f"{name} - Feature Importance")
        ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig("plots/09_feature_importance.png", dpi=120, bbox_inches="tight")
    plt.close()


# ============================================================================
# 11. MAIN
# ============================================================================
if __name__ == "__main__":

    # ---------- Run everything ----------
    if RUN_NAIVE:
        print("\n" + "="*70 + "\nNAIVE BASELINES\n" + "="*70)
        for name, folds in evaluate_naive_baselines(X_all, y_seq).items():
            all_fold_metrics[name] = folds
            summarize(folds, name)

    if RUN_LOGREG:
        print("\n" + "="*70 + "\nLOGISTIC REGRESSION\n" + "="*70)
        all_fold_metrics["LogReg"] = run_sklearn_model(
            lambda: LogisticRegression(class_weight="balanced",
                                       max_iter=2000, C=0.5),
            "LogReg", X_all, y_seq, store_importance_attr="coef_")
        summarize(all_fold_metrics["LogReg"], "LogReg")

    if RUN_RF:
        print("\n" + "="*70 + "\nRANDOM FOREST\n" + "="*70)
        all_fold_metrics["RandomForest"] = run_sklearn_model(
            lambda: RandomForestClassifier(n_estimators=300, max_depth=8,
                                           class_weight="balanced",
                                           random_state=SEED, n_jobs=-1),
            "RandomForest", X_all, y_seq,
            store_importance_attr="feature_importances_")
        summarize(all_fold_metrics["RandomForest"], "RandomForest")

    if RUN_XGB and HAS_XGB:
        print("\n" + "="*70 + "\nXGBOOST\n" + "="*70)
        # Compute scale_pos_weight on full y for class imbalance
        spw = (y_seq == 0).sum() / max(1, (y_seq == 1).sum())
        all_fold_metrics["XGBoost"] = run_sklearn_model(
            lambda: XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
                                  subsample=0.8, colsample_bytree=0.8,
                                  scale_pos_weight=spw,
                                  use_label_encoder=False, eval_metric="logloss",
                                  random_state=SEED, n_jobs=-1, verbosity=0),
            "XGBoost", X_all, y_seq, store_importance_attr="feature_importances_")
        summarize(all_fold_metrics["XGBoost"], "XGBoost")

    if RUN_LGBM and HAS_LGBM:
        print("\n" + "="*70 + "\nLIGHTGBM\n" + "="*70)
        all_fold_metrics["LightGBM"] = run_sklearn_model(
            lambda: LGBMClassifier(n_estimators=400, max_depth=6,
                                   learning_rate=0.05,
                                   class_weight="balanced",
                                   random_state=SEED, n_jobs=-1, verbosity=-1),
            "LightGBM", X_all, y_seq, store_importance_attr="feature_importances_")
        summarize(all_fold_metrics["LightGBM"], "LightGBM")

    keras_models = []
    if RUN_CNN:         keras_models.append((build_cnn,         "CNN"))
    if RUN_GRU:         keras_models.append((build_gru,         "GRU"))
    if RUN_LSTM:        keras_models.append((build_lstm,        "LSTM"))
    if RUN_CNN_LSTM:    keras_models.append((build_cnn_lstm,    "CNN-LSTM"))
    if RUN_TRANSFORMER: keras_models.append((build_transformer, "Transformer"))
    if RUN_TCN:         keras_models.append((build_tcn,         "TCN"))
    if RUN_TFT:         keras_models.append((build_tft,         "TFT"))
    if RUN_NBEATS:      keras_models.append((build_nbeats,      "NBEATS"))

    for builder, name in keras_models:
        print("\n" + "="*70 + f"\n{name}\n" + "="*70)
        all_fold_metrics[name] = run_keras_model(builder, name, X_all, y_seq)
        summarize(all_fold_metrics[name], name)

    # ---------- Build summary tables ----------
    summary_rows = []
    by_fold_rows = []
    for name, folds in all_fold_metrics.items():
        row = {"Model": name}
        for k in METRIC_KEYS:
            vals = [f[k] for f in folds if not np.isnan(f.get(k, np.nan))]
            row[f"{k}_mean"] = np.mean(vals) if vals else np.nan
            row[f"{k}_std"]  = np.std(vals) if vals else np.nan
        # For convenience, also flat columns
        for k in METRIC_KEYS:
            row[k] = row[f"{k}_mean"]
        summary_rows.append(row)
        for fi, fold_m in enumerate(folds, 1):
            by_fold_rows.append({"Model": name, "Fold": fi, **fold_m})

    df_summary = pd.DataFrame(summary_rows)
    df_summary = df_summary.sort_values("MCC_mean", ascending=False).reset_index(drop=True)
    df_by_fold = pd.DataFrame(by_fold_rows)

    # ---------- Save CSVs ----------
    df_summary.to_csv("results_summary.csv", index=False)
    df_by_fold.to_csv("results_by_fold.csv", index=False)

    # ---------- Print summary table ----------
    print("\n" + "="*70 + "\nFINAL SUMMARY (sorted by MCC)\n" + "="*70)
    print_cols = ["Model"] + [f"{m}_mean" for m in METRIC_KEYS]
    print(df_summary[print_cols].round(4).to_string(index=False))

    # ---------- Generate plots ----------
    print("\nGenerating plots in ./plots/ ...")
    plot_metric_bars(df_summary)
    plot_radar_top_models(df_summary)
    plot_per_fold_lines(df_by_fold)
    plot_boxplots(df_by_fold)
    plot_ranking_heatmap(df_summary)
    plot_confusion_matrices(df_summary)
    plot_roc_curves(df_summary)
    plot_calibration(df_summary)
    plot_feature_importance()
    print("Done. See ./plots/ and *.csv files.")

2026-05-04 14:43:11.756101: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-04 14:43:11.767362: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-04 14:43:11.807460: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-04 14:43:11.807488: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-04 14:43:11.807519: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

ImportError: cannot import name 'calibration_curve' from 'sklearn.metrics' (/home/dem1110/.local/lib/python3.11/site-packages/sklearn/metrics/__init__.py)